# ICU Length of Stay (LOS) — Bivariate EDA (Care Unit, Documentation Density, Admission Type)

**Project:** CMPINF2910: Case Studies in Data Science — Assignment 1 (Data Understanding)  
**Student:** Luis Gonzalez  
**Notebook:** `05_icu_bivariate_eda.ipynb`

## Purpose
This notebook extends the univariate LOS exploration by examining **bivariate relationships** that may be relevant for feasibility and hypothesis generation in the MIMIC-IV demo cohort. Specifically:
- LOS differences by **first ICU care unit** (`first_careunit`)
- LOS association with **documentation/measurement density** (e.g., chart-event volume per stay)
- **Prolonged stay rate** (LOS > 8 days) by **admission type**

## Cohort / Grain
- **Grain:** ICU stay-level (`icu.icustays.stay_id`)
- **Outcome:** ICU LOS in days (computed from `outtime - intime` or using `icu.icustays.los`)
- **Prolonged LOS threshold (working definition):** LOS > 8 days

## Inputs / Outputs
### Inputs (from Postgres → exported as CSV)
- Stay-level core extract (LOS + identifiers + first care unit):  
  `data/processed/icu_stay_level_los.csv`  *(exported from SQL in `data/sql/04_extract_icu_los_for_eda.sql`, or your equivalent)*
- Stay-level chart-event counts (events per stay):  
  `data/processed/icu_stay_chartevent_counts.csv`  *(exported from Postgres; query documented in the companion SQL file for this notebook)*
- Admission-level attributes needed for grouping (e.g., `admission_type`):  
  `data/processed/admissions_core.csv`  *(or included in the stay-level extract via join)*

### Output figures (saved for LaTeX report)
- `reports/assignment_1/figures/icu_los_by_first_careunit_boxplot.png`
- `reports/assignment_1/figures/icu_los_vs_chartevents_scatter.png`
- `reports/assignment_1/figures/icu_prolonged_rate_by_admission_type.png`

## Repro Notes
- Dataset is **MIMIC-IV demo**; counts are small and dates are not intended for real-world trend inference.
- All joins rely on validated identifiers: `subject_id`, `hadm_id`, `stay_id`.
- Figures are designed to support the **Data Exploration Report** section of `reports/assignment_1/main.tex`.



In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

repo_root = Path("/home/luis/Documents/mdms_pitt/courses/Capstone/projects/case-studies-capstone").resolve()
data_path = repo_root / "data" / "processed" / "icu_stay_level_features.csv"
fig_dir = repo_root / "reports" / "assignment_1" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(data_path)

In [5]:
# ------------------------------------------
# Figure 1: LOS by first_careunit
# ------------------------------------------

df1 = df.dropna(subset=["first_careunit", "los_days"]).copy()
order = (
    df1.groupby("first_careunit")["los_days"]
       .median()
       .sort_values()
       .index
       .tolist()
)

data = [df1.loc[df1["first_careunit"] == cu, "los_days"].values for cu in order]

plt.figure()
bp = plt.boxplot(data, labels=order, showfliers=False)

# jittered points
for i, cu in enumerate(order, start=1):
    y = df1.loc[df1["first_careunit"] == cu, "los_days"].values
    x = np.random.normal(loc=i, scale=0.06, size=len(y))
    plt.scatter(x, y, alpha=0.5, s=10)

plt.axhline(8.0, linestyle="--")
plt.ylabel("ICU LOS (days)")
plt.title("ICU LOS by first care unit (ordered by median)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(fig_dir / "icu_los_by_first_careunit_boxplot.png", dpi=300)
plt.close()


In [3]:
# ------------------------------------------
# Figure 2: LOS vs chart-event volume (log1p)
# ------------------------------------------
df2 = df.dropna(subset=["los_days", "n_chartevents"]).copy()
x = np.log1p(df2["n_chartevents"].astype(float))
y = df2["los_days"].astype(float)

plt.figure()
plt.scatter(x, y, alpha=0.6, s=16)
plt.axhline(8.0, linestyle="--")
plt.xlabel("log1p(chart events per stay)")
plt.ylabel("ICU LOS (days)")
plt.title("ICU LOS vs chart-event volume (exploratory)")
plt.tight_layout()
plt.savefig(fig_dir / "icu_los_vs_chartevents_scatter.png", dpi=300)
plt.close()

In [4]:
# ----------------------------------------------------
# Figure 3: Prolonged LOS rate (>8d) by admission_type
# ----------------------------------------------------
df3 = df.dropna(subset=["admission_type"]).copy()
g = (
    df3.groupby("admission_type")
       .agg(n=("stay_id", "count"), rate=("prolonged_los_8d", "mean"))
       .reset_index()
)
g = g.sort_values("rate", ascending=False)

plt.figure()
plt.bar(g["admission_type"], g["rate"])
plt.ylabel("Prolonged ICU stay rate (LOS > 8 days)")
plt.title("Prolonged ICU stay rate by admission type (demo cohort)")
plt.xticks(rotation=30, ha="right")
# annotate counts
for i, row in g.reset_index(drop=True).iterrows():
    plt.text(i, row["rate"], f"n={int(row['n'])}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.savefig(fig_dir / "icu_prolonged_rate_by_admission_type.png", dpi=300)
plt.close()

print("Saved figures to:", fig_dir)

Saved figures to: /home/luis/Documents/mdms_pitt/courses/Capstone/projects/case-studies-capstone/reports/assignment_1/figures
